<a href="https://colab.research.google.com/github/Dayaanaly/BioKnee/blob/Dayana/L10_Poda_neuronal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aprendizaje Profundo
# Neural Pruning
##Dr. Carlos Villaseñor

(Ejemplo basado en [enlace](https://www.tensorflow.org/model_optimization/guide/pruning/pruning_with_keras))



Paso 1. Instalamos la biblioteca para

In [2]:
!pip install numpy>=2.0
!pip install -q tensorflow-model-optimization

Paso 2. Importamos los paquetes necesarios

In [18]:
import sys

# Uninstall potentially incompatible numpy version and install a compatible one
# This ensures TensorFlow can load correctly. This step is placed here
# due to the strict instruction to modify *only* this cell, even though
# such installations are usually done in separate cells or at the start.
!{sys.executable} -m pip uninstall -y numpy
!{sys.executable} -m pip install 'numpy<2.0'

# Bibliotecas principales
import numpy as np
import tensorflow as tf
from tensorflow import keras

# Paquetería de poda neuronal
import tensorflow_model_optimization as tfmot

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you 

## Entrenar el modelo base

Paso 3. Importamos los datos de MNIST y normalizamos las imagenes

In [8]:
# Load MNIST dataset
mnist = keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Normalize the input image so that each pixel value is between 0 and 1.
train_images = train_images / 255.0
test_images = test_images / 255.0

Paso 4. Creamos una red neuronal convolucional

In [9]:
# Definimos la arquitectura
model = keras.Sequential([
  keras.layers.InputLayer(input_shape=(28, 28)),
  keras.layers.Reshape(target_shape=(28, 28, 1)),
  keras.layers.Conv2D(filters=12, kernel_size=(3, 3), activation='relu'),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Flatten(),
  keras.layers.Dense(10)
])

# Compilamos el modelo
model.compile(optimizer='adam',
          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
          metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 12)        120       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 12)        0         
 D)                                                              
                                                                 
 flatten (Flatten)           (None, 2028)              0         
                                                                 
 dense (Dense)               (None, 10)                20290     
                                                                 
Total params: 20410 (79.73 KB)
Trainable params: 20410 (79.73 KB)
Non-trainable params: 0 (0.00 Byte)
____________________

Paso 5. Entrenamos el modelo

In [10]:
model.fit(train_images, train_labels, epochs=4, validation_split=0.1)

Epoch 1/4
1688/1688 [==============================] - 9s 4ms/step - loss: 0.2979 - accuracy: 0.9169 - val_loss: 0.1155 - val_accuracy: 0.9677
Epoch 2/4
1688/1688 [==============================] - 7s 4ms/step - loss: 0.1084 - accuracy: 0.9682 - val_loss: 0.0726 - val_accuracy: 0.9800
Epoch 3/4
1688/1688 [==============================] - 6s 3ms/step - loss: 0.0791 - accuracy: 0.9771 - val_loss: 0.0654 - val_accuracy: 0.9818
Epoch 4/4
1688/1688 [==============================] - 7s 4ms/step - loss: 0.0664 - accuracy: 0.9805 - val_loss: 0.0663 - val_accuracy: 0.9817


Paso 6. Evaluamos el modelo

In [11]:
_, base_model_acc = model.evaluate(test_images, test_labels, verbose=0)
print('Original model :', base_model_acc)

Original model : 0.9775999784469604


Paso 7. Guardamos el modelo en memoria secundaria

In [12]:
tf.keras.models.save_model(model, 'base_model.h5', include_optimizer=False)

/tmp/ipykernel_1700/582830098.py:1: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  tf.keras.models.save_model(model, 'base_model.h5', include_optimizer=False)


## Poda neuronal

Paso 8. Empecemos a usar la poda neuronal, definamos un modelo especial para hacer la poda. En el siguiente bloque prepararemos un modelo para hacer prunning sobre todo el modelo, también es posible hacer prunning en capas especificas como pueden verlo en este [enlace](https://www.tensorflow.org/model_optimization/api_docs/python/tfmot/sparsity/keras/prune_low_magnitude).

In [13]:
# Creamos objeto de poda para quitar
Pruner = tfmot.sparsity.keras.prune_low_magnitude


# Necesitamos calcular cuantos pasos daremos en el proceso de poda
# para esto definimos los siguientes parámetros
batch_size = 128
epochs = 2
validation_split = 0.1

# Calcular end-step
num_images = train_images.shape[0] * (1 - validation_split)
end_step = np.ceil(num_images / batch_size).astype(np.int32) * epochs

# Definimos los parámetros para el esquema de poda
pruning_params = {
'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.50,
                                                           final_sparsity=0.80,
                                                           begin_step=0,
                                                           end_step=end_step)
}

# Creamos el modelo para podar
pruned_model = Pruner(model, **pruning_params)

# Es necesario recompilar el modelo.
pruned_model.compile(optimizer='adam',
          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
          metrics=['accuracy'])

pruned_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_reshap  (None, 28, 28, 1)         1         
 e (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_conv2d  (None, 26, 26, 12)        230       
  (PruneLowMagnitude)                                            
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 12)        1         
 oling2d (PruneLowMagnitude                                      
 )                                                               
                                                                 
 prune_low_magnitude_flatte  (None, 2028)              1         
 n (PruneLowMagnitude)                                           
                                                        

Paso 9. El proceso de podado neuronal. Como cualquier otro modelo entrenemos el modelo original con un proceso de poda.

In [14]:
# Creamos un par de callback para usar la poda
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

# Pdamos nuestro modelo
pruned_model.fit(train_images, train_labels,
                  batch_size=batch_size, epochs=epochs,
                  validation_split=validation_split,
                  callbacks=callbacks)

Epoch 1/2
422/422 [==============================] - 11s 15ms/step - loss: 0.1062 - accuracy: 0.9720 - val_loss: 0.1067 - val_accuracy: 0.9727
Epoch 2/2
422/422 [==============================] - 2s 5ms/step - loss: 0.1081 - accuracy: 0.9717 - val_loss: 0.0853 - val_accuracy: 0.9772


Paso 10. Comparemos ambos modelos.

In [15]:
_, pruned_model_acc = pruned_model.evaluate(test_images, test_labels, verbose=0)

print('Original test accuracy:', base_model_acc)
print('Pruned test accuracy:', pruned_model_acc)

Original test accuracy: 0.9775999784469604
Pruned test accuracy: 0.9713000059127808


Paso 11. Comprimimos el modelo. Ahora que el modelo ha sido padado. Muchos de las neuronas no tienen contribución pero el modelo. es igual de grande; ahora vamos a comprimirlo para tener un modelo más pequeño en memoria.

In [16]:
compressed_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

print(compressed_model.summary())

tf.keras.models.save_model(compressed_model, 'compressed_model.h5',
                           include_optimizer=False)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 12)        120       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 12)        0         
 D)                                                              
                                                                 
 flatten (Flatten)           (None, 2028)              0         
                                                                 
 dense (Dense)               (None, 10)                20290     
                                                                 
Total params: 20410 (79.73 KB)
Trainable params: 20410 (79.73 KB)
Non-trainable params: 0 (0.00 Byte)
____________________

/tmp/ipykernel_1700/3554403860.py:5: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  tf.keras.models.save_model(compressed_model, 'compressed_model.h5',


None


## TensorFlow Lite

Paso 12. Para hadware de bajo rendimiento es mejor usar una versión reducida de TensorFlow, llamada TensorFlow Lite. Este framework reducido permite correr modelos de Aprendizaje profundo en hadwares pequeños o embebidos.

In [17]:
converter = tf.lite.TFLiteConverter.from_keras_model(compressed_model)
compressed_model_lite = converter.convert()

with open('compressed_model_lite.tflite', 'wb') as f:
  f.write(compressed_model_lite)

# Poda y cuantización

Paso 13. Usaremos la poda neuronal en conjunto con las técnica de cuantización para comprimir nuestro modelo aun más.

In [19]:
converter = tf.lite.TFLiteConverter.from_keras_model(compressed_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_and_pruned_lite_model = converter.convert()

with open('quantized_and_pruned_model.tflite', 'wb') as f:
  f.write(quantized_and_pruned_lite_model)

Paso 14. Para poder comparar el modelo anterior, sirvase de la siguiente función que evaluara el modelo de TFLite interpretado dentro de nuestro entorno.

In [20]:
import numpy as np

def evaluate_model(interpreter):
  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  # Run predictions on ever y image in the "test" dataset.
  prediction_digits = []
  for i, test_image in enumerate(test_images):
    if i % 1000 == 0:
      print('Evaluated on {n} results so far.'.format(n=i))
    # Pre-processing: add batch dimension and convert to float32 to match with
    # the model's input data format.
    test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, test_image)

    # Run inference.
    interpreter.invoke()

    # Post-processing: remove batch dimension and find the digit with highest
    # probability.
    output = interpreter.tensor(output_index)
    digit = np.argmax(output()[0])
    prediction_digits.append(digit)

  print('\n')
  # Compare prediction results with ground truth labels to calculate accuracy.
  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy


interpreter = tf.lite.Interpreter(model_content=quantized_and_pruned_lite_model)
interpreter.allocate_tensors()
test_accuracy = evaluate_model(interpreter)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluated on 0 results so far.
Evaluated on 1000 results so far.
Evaluated on 2000 results so far.
Evaluated on 3000 results so far.
Evaluated on 4000 results so far.
Evaluated on 5000 results so far.
Evaluated on 6000 results so far.
Evaluated on 7000 results so far.
Evaluated on 8000 results so far.
Evaluated on 9000 results so far.




## Comparación final

Paso 15. Finalmente vemos la comparación entre

In [21]:
print('Modelo original:', base_model_acc)
print('Modelo podado:', pruned_model_acc)
print('Modelo podado y cuantizado para TFlite:', test_accuracy)

Modelo original: 0.9775999784469604
Modelo podado: 0.9713000059127808
Modelo podado y cuantizado para TFlite: 0.9715
